In [1]:
# Импорт библиотек
import pandas as pd
import openpyxl
import numpy as np
import re

# 1. ЗАГРУЗКА ДАННЫХ

In [2]:
df = pd.read_excel("dataset_merged.xlsx")
df.head()

,Название,Цена,URL объявления,Описание,Дата публикации,Продавец,Город,Адрес,Img,URL группы,...,dist_to_park,dist_to_bus_stop,dist_to_supermarket,Субъект РФ,Индекс города,Курортный,"Cредняя зп в городе, тыс руб (2025)",2015 население,Динамика населения за 10 лет,"ВРП района 2023, млн руб"
0,"2-к. квартира, 48,1 м², 17/17 эт.",5964400,https://www.avito.ru//kirovskaya_oblast_kirov/...,"ЖК «Скандинавия», ул. Анжелия Михеева 5.\n\nВ ...",2025-09-22 13:37:50,i198476227,Киров,"ул. Анжелия Михеева, д. 5",https://50.img.avito.st/image/1/1.8_XckLa6Xxyq...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.41,0.25,0.06,Кировская область,233.0,N,69.2,493336.0,-17465.0,605915.5
1,"1-к. квартира, 39,1 м², 11/12 эт.",5829810,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:46:23,i275700606,Киров,"ул. Свободы, д. 141",https://60.img.avito.st/image/1/1.VMI9qba6-CtL...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.22,0.24,0.33,Кировская область,233.0,N,69.2,493336.0,-17465.0,605915.5
2,"3-к. квартира, 52,9 м², 17/17 эт.",6400900,https://www.avito.ru//kirovskaya_oblast_kirov/...,Сохраняем традиции и создаём новые. Слобода — ...,2025-09-29 10:26:27,i198476227,Киров,"ул. Потребкооперации, д. 34, корп. 1",https://50.img.avito.st/image/1/1.8QtqMLa6XeIc...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,1.59,0.45,0.53,Кировская область,233.0,N,69.2,493336.0,-17465.0,605915.5
3,"1-к. квартира, 39 м², 10/12 эт.",5814900,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:37:16,i275700606,Киров,"ул. Свободы, д. 141",https://00.img.avito.st/image/1/1.6FsUTra6RLJi...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.22,0.24,0.33,Кировская область,233.0,N,69.2,493336.0,-17465.0,605915.5
4,"1-к. квартира, 40,1 м², 12/12 эт.",6315750,https://www.avito.ru//kirovskaya_oblast_kirov/...,Дом Сдан! Выберите самую лучшую квартиру для с...,2025-09-29 20:42:24,i275700606,Киров,"ул. Свободы, д. 141",https://00.img.avito.st/image/1/1.BinIDra6qsC-...,https://www.avito.ru/kirovskaya_oblast_kirov/k...,...,0.22,0.24,0.33,Кировская область,233.0,N,69.2,493336.0,-17465.0,605915.5


# 2. ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ИЗ КОЛОНКИ 'НАЗВАНИЕ'

In [3]:
df["Количество_комнат"] = df['Название'].str.split(',').str[0]
df['Количество_комнат'] = df['Количество_комнат'].str.replace('-к. квартира', '').str.strip()

# Площадь 
df['Площадь'] = df['Название'].str.split(',').str[1]
df['Площадь'] = df['Площадь'].str.replace('м²', '').str.strip()

# Этаж и этажность дома 
extracted = df['Название'].str.extract(r'(\d+)\s*/\s*(\d+)\s*эт', expand=True)
extracted.columns = ['Этаж', 'Этажность_дома']
for col in ['Этаж', 'Этажность_дома']:
    extracted[col] = pd.to_numeric(extracted[col], errors='coerce')
df[['Этаж', 'Этажность_дома']] = extracted


# 3. ОЧИСТКА ОТ НЕНУЖНЫХ КОЛОНОК

In [4]:
columns_to_drop = ["Название", "URL объявления", "Описание", "Продавец", 
                   "Адрес", "Img", "URL группы", "Unnamed: 10", "Дата парсинга"]
df_clear = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
print(f" Удалено {len([col for col in columns_to_drop if col in df.columns])} колонок")


 Удалено 9 колонок


# 4. ОБРАБОТКА ПРОПУСКОВ

In [5]:
missing_before = df_clear.isnull().sum().sum()


if missing_before > 0:
    rows_before = len(df_clear)
    df_clear = df_clear.dropna()
    print(f"Удалено строк с пропусками: {rows_before - len(df_clear)}")
else:
    print("Пропусков нет")

Удалено строк с пропусками: 256


# 5. ПРЕОБРАЗОВАНИЕ ТИПОВ ДАННЫХ

In [6]:
# Обработка запятых и преобразование в int
df_clear['Площадь'] = df_clear['Площадь'].str.replace(',', '.').astype(float).round().astype(int)

# Цена в int
df_clear['Цена'] = df_clear['Цена'].astype(int)

# Создаем целевую переменную
df_clear['Цена_за_квадратный_метр'] = df_clear['Цена'] / df_clear['Площадь']

# Курортный: N/Y -> False/True
df_clear['Курортный'] = df_clear['Курортный'].map({'N': False, 'Y': True})

# Преобразование в числовой формат
df_clear['ВРП района 2023, млн руб'] = pd.to_numeric(
    df_clear['ВРП района 2023, млн руб'], errors='coerce'
)
median_vrp = df_clear['ВРП района 2023, млн руб'].median()
df_clear['ВРП района 2023, млн руб'] = df_clear['ВРП района 2023, млн руб'].fillna(median_vrp)



# 6. ОБРАБОТКА КОЛИЧЕСТВА КОМНАТ И ТИПА НЕДВИЖИМОСТИ

In [7]:
# Функция для извлечения числа комнат
def get_rooms_count(text):
    text_lower = str(text).lower()
    
    if 'студия' in text_lower:
        return 0
    if 'своб. планировка' in text_lower:
        return 1
    if '10 и более' in text_lower:
        return 10
    
    match = re.search(r'^(\d+)', text_lower)
    if match:
        return int(match.group(1))
    
    return np.nan
df_clear['Rooms_Count'] = df_clear['Количество_комнат'].apply(get_rooms_count)

In [8]:
# Определяем тип недвижимости
conditions = [
    df_clear['Количество_комнат'].str.contains('студия', case=False, na=False),
    df_clear['Количество_комнат'].str.contains('Своб. планировка', case=False, na=False),
    df_clear['Количество_комнат'].str.contains('апартаменты', case=False, na=False)
]
choices = ['Студия', 'Своб. планировка', 'Апартаменты']
df_clear['Property_Type'] = np.select(conditions, choices, default='Квартира')


print(f"Распределение типов: {df_clear['Property_Type'].value_counts().to_dict()}")

# One-Hot Encoding
df_final = pd.get_dummies(df_clear, columns=['Property_Type'], drop_first=True, dtype=int)
df_final = df_final.drop(columns=['Количество_комнат'])

print(f"One-hot encoding выполнен")
print(f"Созданные колонки: {[col for col in df_final.columns if 'Property_Type' in col]}")

Распределение типов: {'Квартира': 17059, 'Студия': 1133, 'Своб. планировка': 208, 'Апартаменты': 166}
One-hot encoding выполнен
Созданные колонки: ['Property_Type_Квартира', 'Property_Type_Своб. планировка', 'Property_Type_Студия']


# 7. УДАЛЕНИЕ ДУБЛИКАТОВ

In [9]:
duplicates = df_final.duplicated().sum()
if duplicates > 0:
    df_final = df_final.drop_duplicates()
    print(f"Удалено дубликатов: {duplicates}")
else:
    print("Дубликатов нет")

Удалено дубликатов: 252


# 8. ОБРАБОТКА ДАТЫ ПУБЛИКАЦИИ

In [10]:
df_final['Дата публикации'] = pd.to_datetime(df_final['Дата публикации'], errors='coerce')
df_final['Year_Public'] = df_final['Дата публикации'].dt.year
df_final['Month_Public'] = df_final['Дата публикации'].dt.month
df_final['DayOfWeek_Public'] = df_final['Дата публикации'].dt.dayofweek

# 9. СОЗДАНИЕ ДОПОЛНИТЕЛЬНЫХ ПРИЗНАКОВ

In [11]:
# Доля этажа от общей этажности
df_final['Floor_Ratio'] = df_final['Этаж'] / df_final['Этажность_дома']

# Первый и последний этаж
df_final['Is_First_Floor'] = (df_final['Этаж'] == 1).astype(int)
df_final['Is_Last_Floor'] = (df_final['Этаж'] == df_final['Этажность_дома']).astype(int)

# Площадь на комнату
df_final['Area_per_Room'] = df_final['Площадь'] / df_final['Rooms_Count'].replace(0, 1)

# Индекс инфраструктуры
infrastructure_cols = ['dist_to_school', 'dist_to_kindergarten', 'dist_to_park', 
                       'dist_to_bus_stop', 'dist_to_supermarket']
df_final['Infrastructure_Score'] = df_final[infrastructure_cols].sum(axis=1)

# 10. ОБРАБОТКА ВЫБРОСОВ

In [12]:
initial_size = len(df_final)

# Цена
price_limit = 30_000_000
before = len(df_final)
df_final = df_final[df_final['Цена'] <= price_limit]

In [13]:
# Расстояние от центра
max_dist = 200
before = len(df_final)
df_final = df_final[df_final['dist_to_city_center'] <= max_dist]

In [14]:
# Площадь (минимум)
min_area = 10
before = len(df_final)
df_final = df_final[df_final['Площадь'] >= min_area]

In [15]:
# Площадь (максимум)
max_area = 200
before = len(df_final)
df_final = df_final[df_final['Площадь'] <= max_area]

In [16]:
max_price = 23000000
before = len(df_final)
df_final = df_final[df_final['Цена'] <= max_price]


In [17]:
# Цена за кв.м (выбросы)
q01 = df_final['Цена_за_квадратный_метр'].quantile(0.01)
q99 = df_final['Цена_за_квадратный_метр'].quantile(0.99)
before = len(df_final)
df_final = df_final[
    (df_final['Цена_за_квадратный_метр'] >= q01) & 
    (df_final['Цена_за_квадратный_метр'] <= q99)
]

In [18]:
print(f"Всего удалено выбросов: {initial_size - len(df_final)}")

Всего удалено выбросов: 805


# 11. ПРЕОБРАЗОВАНИЕ BOOL В INT

In [19]:
bool_cols = df_final.select_dtypes(include='bool').columns
if len(bool_cols) > 0:
    df_final[bool_cols] = df_final[bool_cols].astype(int)
    print(f"Преобразовано {len(bool_cols)} булевых колонок в int")
else:
    print("Булевых колонок не найдено")

Преобразовано 33 булевых колонок в int


# 12. ЛОГАРИФМИЧЕСКИЕ ПРЕОБРАЗОВАНИЯ

In [20]:
# Создаем новые колонки с логарифмами 
df_final['Площадь_log'] = np.log1p(df_final['Площадь'])
df_final['Цена_log'] = np.log1p(df_final['Цена'])
df_final['Цена_за_квадратный_метр_log'] = np.log1p(df_final['Цена_за_квадратный_метр'])

# 13. ИТОГОВАЯ ПРОВЕРКА

In [21]:

print("ИТОГОВАЯ СТАТИСТИКА")


print(f"Исходный размер: {df.shape[0]} строк, {df.shape[1]} колонок")
print(f"Финальный размер: {df_final.shape[0]} строк, {df_final.shape[1]} колонок")
print(f"Удалено строк: {df.shape[0] - df_final.shape[0]} ")

print(f"\nПроверки:")
print(f"Пропуски: {df_final.isnull().sum().sum()}")
print(f"Дубликаты: {df_final.duplicated().sum()}")
print(f"Inf значения: {np.isinf(df_final.select_dtypes(include=[np.number])).sum().sum()}")

print(f"\nКлючевые статистики:")
print(df_final[['Цена', 'Площадь', 'Rooms_Count', 'Этаж', 'Этажность_дома', 
                'Цена_за_квадратный_метр']].describe())

ИТОГОВАЯ СТАТИСТИКА
Исходный размер: 18822 строк, 61 колонок
Финальный размер: 17509 строк, 67 колонок
Удалено строк: 1313 

Проверки:
Пропуски: 0
Дубликаты: 0
Inf значения: 0

Ключевые статистики:
               Цена       Площадь   Rooms_Count          Этаж  Этажность_дома  \
count  1.750900e+04  17509.000000  17509.000000  17509.000000    17509.000000   
mean   7.163112e+06     54.433092      1.640014      6.299846       12.100577   
std    3.152570e+06     19.453668      0.844556      4.411169        5.045848   
min    1.059500e+06     13.000000      0.000000      1.000000        1.000000   
25%    4.918080e+06     40.000000      1.000000      3.000000        9.000000   
50%    6.487000e+06     51.000000      2.000000      5.000000       10.000000   
75%    8.694000e+06     66.000000      2.000000      9.000000       16.000000   
max    2.295900e+07    186.000000      4.000000     37.000000       45.000000   

       Цена_за_квадратный_метр  
count             17509.000000  
mean  

In [22]:
df_final.describe()

,Цена,Дата публикации,is_class_eco,is_class_comfort,is_class_business,is_class_elite,is_brick,is_monolith,is_panel,is_new_build,...,Month_Public,DayOfWeek_Public,Floor_Ratio,Is_First_Floor,Is_Last_Floor,Area_per_Room,Infrastructure_Score,Площадь_log,Цена_log,Цена_за_квадратный_метр_log
count,1.750900e+04,17509,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,...,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000,17509.000000
mean,7.163112e+06,2025-10-07 21:54:11.775486976,0.000057,0.363584,0.112114,0.030327,0.233366,0.171340,0.018562,0.553544,...,9.735507,2.501514,0.532864,0.085727,0.091724,34.331177,2.791786,3.957135,15.697896,11.761237
min,1.059500e+06,2025-06-25 19:12:43,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,6.000000,0.000000,0.040000,0.000000,0.000000,13.000000,0.670000,2.639057,13.873309,11.002117
25%,4.918080e+06,2025-09-30 10:37:10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.000000,1.000000,0.272727,0.000000,0.000000,28.000000,1.440000,3.713572,15.408429,11.561267
50%,6.487000e+06,2025-10-08 14:40:31,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,10.000000,2.000000,0.500000,0.000000,0.000000,33.000000,2.040000,3.951244,15.685311,11.725769
75%,8.694000e+06,2025-10-15 18:26:37,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,...,10.000000,4.000000,0.777778,0.000000,0.000000,39.500000,3.490000,4.204693,15.978144,11.933514
max,2.295900e+07,2025-10-21 15:37:25,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,10.000000,6.000000,1.000000,1.000000,1.000000,132.000000,35.610000,5.231109,16.949221,12.845792
std,3.152570e+06,NaN,0.007557,0.481045,0.315515,0.171491,0.422985,0.376817,0.134975,0.497139,...,0.441981,1.976519,0.284978,0.279969,0.288645,9.167870,2.198857,0.339502,0.412468,0.288430


In [23]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17509 entries, 0 to 18821
Data columns (total 67 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   Цена                                 17509 non-null  int64         
 1   Дата публикации                      17509 non-null  datetime64[ns]
 2   Город                                17509 non-null  object        
 3   is_class_eco                         17509 non-null  int64         
 4   is_class_comfort                     17509 non-null  int64         
 5   is_class_business                    17509 non-null  int64         
 6   is_class_elite                       17509 non-null  int64         
 7   is_brick                             17509 non-null  int64         
 8   is_monolith                          17509 non-null  int64         
 9   is_panel                             17509 non-null  int64         
 10  is_new_build   

In [24]:
df_final

,Цена,Дата публикации,Город,is_class_eco,is_class_comfort,is_class_business,is_class_elite,is_brick,is_monolith,is_panel,...,Month_Public,DayOfWeek_Public,Floor_Ratio,Is_First_Floor,Is_Last_Floor,Area_per_Room,Infrastructure_Score,Площадь_log,Цена_log,Цена_за_квадратный_метр_log
0,5964400,2025-09-22 13:37:50,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,24.000000,3.42,3.891820,15.601319,11.730126
1,5829810,2025-09-29 20:46:23,Киров,0,0,0,0,0,0,0,...,9,0,0.916667,0,0,39.000000,1.26,3.688879,15.578495,11.914940
2,6400900,2025-09-29 10:26:27,Киров,0,1,0,0,0,0,0,...,9,0,1.000000,0,1,17.333333,4.01,3.970292,15.671949,11.720714
3,5814900,2025-09-29 20:37:16,Киров,0,0,0,0,0,0,0,...,9,0,0.833333,0,0,39.000000,1.26,3.688879,15.575934,11.912379
4,6315750,2025-09-29 20:42:24,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,40.000000,1.26,3.713572,15.658557,11.969684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18783,6336000,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,1.000000,0,1,32.000000,2.70,4.174387,15.661758,11.502885
18788,6399400,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,0.666667,0,0,32.000000,2.70,4.174387,15.671715,11.512842
18790,6252400,2025-10-16 15:39:56,Гагарин,0,0,1,0,0,1,0,...,10,3,0.666667,0,0,31.500000,2.70,4.158883,15.648476,11.505351
18796,6316200,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,0.888889,0,0,31.500000,2.70,4.158883,15.658628,11.515504


# 14. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ

In [25]:
df_final.to_excel('df_final.xlsx', index=False)

In [26]:
df_final.to_csv('df_final.csv', index=False)

Удалить Сочи как выброс

Сделать график для цена за кв метр лог
